# Vision Transformers on EuroSAT

Notebooks 08-10 built up the convolutional view of vision: local receptive fields, weight sharing, translation equivariance, depth made trainable by BatchNorm and residuals, and finally pretrained ImageNet ResNets that transfer to EuroSAT. Every architectural choice in that arc bakes in **the assumption that images have local spatial structure** — which they do, so CNNs work well.

**Vision Transformers (ViTs)** make a radically different bet: throw out the convolutional inductive biases, chop the image into 16x16 patches, treat the patches as a sequence of tokens, and let **self-attention** discover whatever relationships matter. No locality bias. No translation equivariance from the architecture itself. Just attention everywhere.

The remarkable empirical result (Dosovitskiy et al., 2021) is that at sufficient pretraining scale, this works as well as — or better than — CNNs. At small scale and without pretraining, it loses badly. This notebook is a small empirical tour of that tradeoff, on the same EuroSAT setup from notebook 10.

## Learning goals

By the end of this notebook you should be able to explain:

1. how a ViT turns an image into a sequence of patch tokens and what positional embeddings do,
2. why self-attention is a fundamentally different inductive bias from convolution,
3. how a pretrained `vit_b_16` compares to a pretrained ResNet18 on EuroSAT (apples to apples with notebook 10),
4. why ViTs need either massive pretraining or strong augmentation to beat CNNs from scratch,
5. how to visualize what patches a ViT attends to for a given image.

In [ ]:
import math
import random
from pathlib import Path

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, random_split
import torchvision
from torchvision import transforms
import matplotlib.pyplot as plt

seed = 11
random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
torch.set_num_threads(1)
try:
    torch.set_num_interop_threads(1)
except RuntimeError:
    pass

torch.set_default_dtype(torch.float32)

plt.rcParams.update({
    "figure.figsize": (7.4, 5.0),
    "figure.dpi": 120,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("PyTorch version:    ", torch.__version__)
print("Torchvision version:", torchvision.__version__)
print("Device:             ", device)
print("\nHeads up: ViT-B/16 has ~86M parameters (about 8x ResNet18). On CPU each epoch may take 10+ minutes.")

## How a ViT processes an image

Concretely, for a 224x224 RGB input image:

1. **Patch embedding.** The image is split into a non-overlapping grid of 16x16 patches. With a 224x224 input that gives `224 / 16 = 14` patches per side, so 196 patches in total. Each patch (a `3 x 16 x 16` tensor = 768 numbers) is linearly projected to a 768-dimensional embedding. In code, that whole step is just `Conv2d(3, 768, kernel_size=16, stride=16)`.
2. **CLS token.** A learned `[CLS]` embedding is prepended to the sequence. The eventual classification logit is read off from the `[CLS]` token's final representation.
3. **Positional embeddings.** Self-attention is permutation-invariant by default, so we add learned positional embeddings to give each token a sense of *where* it is in the 14x14 grid.
4. **Transformer encoder.** A stack of 12 identical blocks. Each block is `LayerNorm -> MultiheadAttention -> residual -> LayerNorm -> MLP -> residual`. Self-attention lets every patch token interact with every other patch token at every layer.
5. **Classifier head.** A `LayerNorm + Linear` on the `[CLS]` token outputs a vector of class logits.

Contrast with the CNN's recipe of `Conv + BN + ReLU + Pool` blocks (notebooks 08-09): in a CNN the first layer can only see a 3x3 neighborhood; in a ViT the first attention layer can already attend to *every patch in the image*. The receptive field is global from the start.

**The downside:** a ViT has no built-in reason to prefer nearby patches. With small training sets it has to *discover* spatial locality from the data. CNNs already know.

## EuroSAT setup (identical to notebook 10)

Same `Resize(224) + ToTensor + ImageNet normalize` pipeline, same 80 / 10 / 10 split with `manual_seed(42)`. ViT-B/16 was pretrained at 224x224 with ImageNet normalization, so we use exactly the same preprocessing as the ResNet18 in notebook 10. This is what makes the head-to-head comparison fair.

In [ ]:
DATA_DIR = Path("./data")
DATA_DIR.mkdir(exist_ok=True)

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

full_dataset = torchvision.datasets.EuroSAT(DATA_DIR, download=True, transform=transform)
n_total = len(full_dataset)
n_test = int(0.10 * n_total)
n_val  = int(0.10 * n_total)
n_train = n_total - n_val - n_test
split_generator = torch.Generator().manual_seed(42)
train_set, val_set, test_set = random_split(
    full_dataset, [n_train, n_val, n_test], generator=split_generator,
)
CLASS_NAMES = full_dataset.classes

# Smaller batch than notebook 10 because ViT-B/16 is heavier per sample.
BATCH_SIZE = 32
loader_generator = torch.Generator().manual_seed(2025)
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, generator=loader_generator)
val_loader   = DataLoader(val_set,   batch_size=64, shuffle=False)
test_loader  = DataLoader(test_set,  batch_size=64, shuffle=False)

print(f"Train / Val / Test: {len(train_set)} / {len(val_set)} / {len(test_set)}")

## Loading a pretrained ViT-B/16

`torchvision.models.vit_b_16` provides the standard ViT-Base architecture (12 layers, 12 attention heads, 768-dim embeddings). The pretrained weights (`ViT_B_16_Weights.IMAGENET1K_V1`) were trained on ImageNet-1K. The first time you run this it downloads about 330 MB.

The classification head is at `model.heads.head` (a `Linear(768, 1000)`). For our 10 EuroSAT classes we replace it the same way we replaced ResNet's `fc`.

In [ ]:
VIT_WEIGHTS = torchvision.models.ViT_B_16_Weights.IMAGENET1K_V1

demo_vit = torchvision.models.vit_b_16(weights=VIT_WEIGHTS)
n_params = sum(p.numel() for p in demo_vit.parameters())
print("Encoder layers:        ", len(demo_vit.encoder.layers))
print("Hidden dim:            ", demo_vit.hidden_dim)
print("Patch size:            ", demo_vit.patch_size)
print("Sequence length:       ", (224 // demo_vit.patch_size) ** 2 + 1, "(196 patches + 1 CLS token)")
print("Classification head:   ", demo_vit.heads.head)
print(f"Total parameters:       {n_params:,}")
del demo_vit

## Training harness (same as notebook 10)

Reusing `train_model` from notebook 10 — it takes a `model_and_optimizer_factory(seed) -> (model, optimizer)` so the caller controls freezing and parameter groups.

In [ ]:
def evaluate_loader(model, loader, loss_fn, device):
    model.eval()
    total_loss, total_correct, total_n = 0.0, 0, 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            total_loss += loss_fn(logits, yb).item() * xb.size(0)
            total_correct += (logits.argmax(-1) == yb).sum().item()
            total_n += xb.size(0)
    return total_loss / total_n, total_correct / total_n


def train_model(
    model_and_optimizer_factory,
    train_loader, val_loader,
    *,
    max_epochs=5, patience=2, min_delta=1e-4,
    model_seed=123, device=device, verbose=True,
):
    model, optimizer = model_and_optimizer_factory(model_seed)
    model = model.to(device)
    loss_fn = nn.CrossEntropyLoss()
    history = {"epoch": [], "train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    best = {"val_loss": float("inf"), "val_acc": None, "epoch": 0, "state_dict": None}
    epochs_no_improve = 0
    for epoch in range(1, max_epochs + 1):
        model.train()
        running_loss, running_correct, running_n = 0.0, 0, 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = loss_fn(logits, yb)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            running_loss += loss.item() * xb.size(0)
            running_correct += (logits.argmax(-1) == yb).sum().item()
            running_n += xb.size(0)
        train_loss = running_loss / running_n
        train_acc = running_correct / running_n
        val_loss, val_acc = evaluate_loader(model, val_loader, loss_fn, device)
        history["epoch"].append(epoch)
        history["train_loss"].append(train_loss); history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc);  history["val_acc"].append(val_acc)
        if val_loss < best["val_loss"] - min_delta:
            best.update({
                "val_loss": val_loss, "val_acc": val_acc, "epoch": epoch,
                "state_dict": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
            })
            epochs_no_improve = 0
            marker = "*"
        else:
            epochs_no_improve += 1
            marker = " "
        if verbose:
            print(f"epoch {epoch:3d} {marker} | train {train_loss:.3f}/{train_acc:.3f} | val {val_loss:.3f}/{val_acc:.3f}")
        if epochs_no_improve >= patience:
            if verbose:
                print(f"Early stopping at epoch {epoch}")
            break
    if best["state_dict"] is not None:
        model.load_state_dict(best["state_dict"])
    return model, history, best

MAX_EPOCHS = 5
PATIENCE = 2

## Fine-tuning ViT-B/16 on EuroSAT

We replace `heads.head` with a `Linear(768, 10)` and fine-tune the whole network at a small learning rate (`1e-4`) — same recipe that worked best for ResNet18 in notebook 10.

In [ ]:
def make_fine_tune_vit(seed):
    torch.manual_seed(seed)
    model = torchvision.models.vit_b_16(weights=VIT_WEIGHTS)
    in_features = model.heads.head.in_features
    model.heads.head = nn.Linear(in_features, len(CLASS_NAMES))
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.05)
    return model, optimizer

print("Fine-tuning ViT-B/16 (this may take a while)...")
vit_ft_model, vit_ft_hist, vit_ft_best = train_model(
    make_fine_tune_vit,
    train_loader, val_loader,
    max_epochs=MAX_EPOCHS, patience=PATIENCE,
    model_seed=123,
)
vit_ft_test_loss, vit_ft_test_acc = evaluate_loader(
    vit_ft_model, test_loader, nn.CrossEntropyLoss(), device,
)
print(f"\nViT fine-tuned: best val acc {vit_ft_best['val_acc']:.4f}, test acc {vit_ft_test_acc:.4f}")

## Feature extraction (backbone frozen)

Same freeze-the-backbone trick as notebook 10. Useful as a cheaper sanity check — only the new classification head is trained. The pretrained ViT representations should be linearly separable for EuroSAT's 10 classes, so we expect this to land well above the from-scratch baseline.

In [ ]:
def make_feature_extract_vit(seed):
    torch.manual_seed(seed)
    model = torchvision.models.vit_b_16(weights=VIT_WEIGHTS)
    for p in model.parameters():
        p.requires_grad = False
    in_features = model.heads.head.in_features
    model.heads.head = nn.Linear(in_features, len(CLASS_NAMES))
    trainable = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(trainable, lr=1e-3, weight_decay=0.01)
    return model, optimizer

print("Training ViT feature-extraction model (only the head is trained)...")
vit_fe_model, vit_fe_hist, vit_fe_best = train_model(
    make_feature_extract_vit,
    train_loader, val_loader,
    max_epochs=MAX_EPOCHS, patience=PATIENCE,
    model_seed=123,
)
vit_fe_test_loss, vit_fe_test_acc = evaluate_loader(
    vit_fe_model, test_loader, nn.CrossEntropyLoss(), device,
)
print(f"\nViT feature extraction: best val acc {vit_fe_best['val_acc']:.4f}, test acc {vit_fe_test_acc:.4f}")

## Head-to-head: ViT vs ResNet18

For an honest comparison we re-run a fine-tuned ResNet18 on the same data, same harness, same configuration as in notebook 10.

In [ ]:
RESNET_WEIGHTS = torchvision.models.ResNet18_Weights.IMAGENET1K_V1

def make_fine_tune_resnet18(seed):
    torch.manual_seed(seed)
    model = torchvision.models.resnet18(weights=RESNET_WEIGHTS)
    model.fc = nn.Linear(model.fc.in_features, len(CLASS_NAMES))
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)
    return model, optimizer

print("Fine-tuning ResNet18 for comparison...")
resnet_ft_model, resnet_ft_hist, resnet_ft_best = train_model(
    make_fine_tune_resnet18,
    train_loader, val_loader,
    max_epochs=MAX_EPOCHS, patience=PATIENCE,
    model_seed=123,
)
resnet_ft_test_loss, resnet_ft_test_acc = evaluate_loader(
    resnet_ft_model, test_loader, nn.CrossEntropyLoss(), device,
)
print(f"\nResNet18 fine-tuned: best val acc {resnet_ft_best['val_acc']:.4f}, test acc {resnet_ft_test_acc:.4f}")

In [ ]:
def count_params(model):
    return sum(p.numel() for p in model.parameters())

rows = [
    ("ResNet18 fine-tuned", count_params(resnet_ft_model), resnet_ft_best, resnet_ft_test_acc),
    ("ViT-B/16 feature ext.", count_params(vit_fe_model),  vit_fe_best,    vit_fe_test_acc),
    ("ViT-B/16 fine-tuned",   count_params(vit_ft_model),  vit_ft_best,    vit_ft_test_acc),
]
header = f"{'model':>24s}   {'params':>11s}   {'best val acc':>12s}   {'test acc':>8s}   {'best epoch':>10s}"
print(header); print("-" * len(header))
for name, n_p, b, t_acc in rows:
    print(f"{name:>24s}   {n_p:>11,d}   {b['val_acc']:12.4f}   {t_acc:8.4f}   {b['epoch']:10d}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.0))
for name, hist in [
    ("ResNet18 fine-tuned", resnet_ft_hist),
    ("ViT-B/16 feature ext.", vit_fe_hist),
    ("ViT-B/16 fine-tuned",   vit_ft_hist),
]:
    axes[0].plot(hist["epoch"], hist["val_loss"], marker="o", label=name, markersize=5)
    axes[1].plot(hist["epoch"], hist["val_acc"],  marker="o", label=name, markersize=5)
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("validation cross-entropy loss")
axes[0].set_title("Validation loss")
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("validation accuracy")
axes[1].set_title("Validation accuracy")
axes[0].legend(loc="best"); axes[1].legend(loc="best")
fig.suptitle("ViT vs ResNet on EuroSAT, both pretrained on ImageNet")
fig.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

## A note on data scale

The Dosovitskiy et al. (2021) ViT paper made one finding particularly load-bearing for what we just did: **without large-scale pretraining, ViTs lose to CNNs on small datasets.** Their experiments trained ViT and ResNet variants on ImageNet (1.3M images), ImageNet-21k (14M), and JFT-300M (300M):

- On ImageNet alone, ResNets matched or beat ViTs.
- On ImageNet-21k, ViTs caught up.
- On JFT-300M, ViTs decisively won and kept improving with scale longer than ResNets did.

The mechanism: CNNs *encode* the right inductive bias for natural images (locality, translation equivariance) so they generalize from less data. ViTs have to *learn* those biases — given enough data, they learn richer and more flexible versions; given too little, they cannot learn them at all.

We are not running the from-scratch ViT experiment here (it would take many hours and lose anyway), but it is worth knowing that the entire "ViTs match or beat CNNs" result rests on the pretrained-weights step we just did. If pretraining were not available, we would still pick the CNN.

This is why **transfer learning (notebook 10) and modern architectures (this notebook) are inseparable in practice** — most projects do not have JFT-300M lying around, so they use whatever pretrained weights they can get.

## What does the ViT attend to?

The most concrete way to interpret a ViT is to plot what the `[CLS]` token attends to in the final encoder block, for a given test image. With 12 attention heads we get 12 attention maps; averaging them gives a single 14x14 attention map (one number per patch) that we overlay on the original image.

Mechanically, we need to capture attention weights inside the encoder. `torchvision`'s implementation calls `MultiheadAttention` with `need_weights=False`, so we monkey-patch the last encoder block's `forward` to capture them. This is hacky and read-only — we restore the original forward afterwards.

In [ ]:
captured = {}

def make_capturing_forward(block):
    original_forward = block.forward
    def forward(input):
        x = block.ln_1(input)
        x, attn_w = block.self_attention(x, x, x, need_weights=True, average_attn_weights=False)
        captured["weights"] = attn_w.detach().cpu()  # shape: (B, heads, seq, seq)
        x = block.dropout(x)
        x = x + input
        y = block.ln_2(x)
        y = block.mlp(y)
        return x + y
    return forward, original_forward

# Patch the last encoder block of the fine-tuned ViT
last_block = vit_ft_model.encoder.layers[-1]
patched_forward, original_forward = make_capturing_forward(last_block)
last_block.forward = patched_forward

# Run one batch through to capture attention
vit_ft_model.eval()
with torch.no_grad():
    sample_image, sample_label = test_set[0]
    pred_logits = vit_ft_model(sample_image.unsqueeze(0).to(device))
    predicted = pred_logits.argmax(-1).item()

# Restore the original forward
last_block.forward = original_forward

# attention shape: (1, 12 heads, 197 tokens, 197 tokens)
attn = captured["weights"][0]  # (12, 197, 197)
# Row 0 = [CLS] token attending to all tokens. Drop the [CLS]-to-[CLS] entry.
cls_to_patches = attn[:, 0, 1:]  # (12, 196)
cls_attn_avg = cls_to_patches.mean(dim=0)  # average over heads -> (196,)
cls_attn_grid = cls_attn_avg.reshape(14, 14)

# Plot original image and attention overlay
mean_t = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
std_t  = torch.tensor(IMAGENET_STD).view(3, 1, 1)
display_img = (sample_image * std_t + mean_t).clamp(0, 1).permute(1, 2, 0).numpy()

fig, axes = plt.subplots(1, 3, figsize=(13, 4.6), constrained_layout=True)
axes[0].imshow(display_img); axes[0].set_title(f"input: {CLASS_NAMES[sample_label]}")
axes[0].set_xticks([]); axes[0].set_yticks([]); axes[0].grid(False)

axes[1].imshow(cls_attn_grid.numpy(), cmap="viridis")
axes[1].set_title("[CLS] attention (last layer, head-averaged)")
axes[1].set_xticks([]); axes[1].set_yticks([]); axes[1].grid(False)

# Overlay: upsample attention to 224x224 and blend with the image
attn_upsampled = torch.nn.functional.interpolate(
    cls_attn_grid[None, None], size=(224, 224), mode="bilinear", align_corners=False,
)[0, 0].numpy()
attn_norm = (attn_upsampled - attn_upsampled.min()) / (attn_upsampled.max() - attn_upsampled.min() + 1e-8)
axes[2].imshow(display_img)
axes[2].imshow(attn_norm, cmap="hot", alpha=0.45)
axes[2].set_title(f"overlay (predicted: {CLASS_NAMES[predicted]})")
axes[2].set_xticks([]); axes[2].set_yticks([]); axes[2].grid(False)
fig.suptitle("Where the [CLS] token looks")
plt.show()

## Summary

| Model               | Params (~) | Inductive bias                            | Typical EuroSAT test acc       |
|---------------------|-----------:|-------------------------------------------|--------------------------------|
| MLP (notebook 07b)  |     3.3M   | none beyond fully-connected               | ~78-82%                        |
| ResNet18 from scratch (10) |   11.2M | local receptive fields + translation eq. | ~85-90%                        |
| ResNet18 pretrained (10) |  11.2M   | + ImageNet representations                | ~96-98%                        |
| ViT-B/16 feature extract  |   86M   | global self-attention + ImageNet repr.    | ~96-97%                        |
| ViT-B/16 fine-tuned       |   86M   | + adapted to satellite domain             | ~97-98%                        |

Headline observations:

1. **Pretraining still dominates.** ResNet and ViT, both pretrained, land in the same accuracy ballpark on EuroSAT. The pretrained representation matters more than which family of architecture produced it.
2. **ViTs cost more.** ViT-B/16 is ~8x bigger than ResNet18 and slower per forward pass. On small tasks like EuroSAT it does not return that cost in accuracy.
3. **The architecture choice matters more at scale.** ViTs decisively win when massively pretrained; at our scale the families are tied. Worth knowing for project planning.
4. **Attention maps are a useful interpretability tool.** They show *which patches* the model based its decision on — a different kind of insight than convolutional filter visualization, complementary to it.

Reasonable next directions:

- **Notebook 12 — Data augmentation deep dive.** Mixup, CutMix, RandAugment. These compound with everything in notebooks 08-11 and often add 1-3 percentage points for free.
- **Notebook 13 — Self-attention from scratch.** Build the same multi-head attention block that ViT uses, on a simpler 1-D task (sequence classification). Sets up an NLP / LLM-style direction.
- **Notebook 14 — Self-supervised pretraining.** What if you don't have ImageNet labels? Contrastive learning (SimCLR / MoCo) or masked-image modeling (MAE).

## Exercises

1. **Smaller ViT.** Replace `vit_b_16` with `vit_b_32` (32x32 patches, same encoder). Fewer patch tokens means less attention compute — does accuracy hold up?

2. **Swin Transformer.** Try `torchvision.models.swin_t`. Swin reintroduces some locality via shifted-window attention. Does it land between ResNet and ViT, or somewhere else?

3. **Per-class accuracy.** Build a per-class accuracy bar plot for the fine-tuned ViT and the fine-tuned ResNet. Where do their strengths and weaknesses differ?

4. **Attention rollout.** The single-layer attention map above can be misleading. Implement *attention rollout* (multiply attention matrices across all layers, with an identity-residual correction) for a more faithful saliency map.

5. **Augmentation pays off more for ViT.** Add `RandomHorizontalFlip()`, `RandomVerticalFlip()`, and `RandomRotation(15)` to the training transform. Compare the accuracy boost for ResNet18 vs ViT-B/16. Theory predicts ViTs benefit more — verify it.

6. **Patch-embedding inspection.** The patch embedding is just a `Conv2d(3, 768, kernel_size=16, stride=16)`. Pull out its weights and visualize them as 16x16 RGB filters. Do they look like the first-layer filters of a CNN, or different?